# GTU Mimari Lejant - Colab Training

Bu notebook dataset/weight/test Drive yapisini iki ayri linkten alir; YOLO segmentation egitimi, kayitli weight ile test ve rapor export akisini calistirir. GPU onceligi: A100 > L4 > T4.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Repo'yu Klonla

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
%cd /content
!rm -rf /content/lejanter_doga_vlm_codex
!git clone https://github.com/doganalci/lejanter_doga_vlm_codex.git /content/lejanter_doga_vlm_codex
%cd /content/lejanter_doga_vlm_codex
!pip install -q -r requirements.txt

## 2. Drive'dan Test Gorselleri ve Weight Dosyalarini Al

Test klasoru ile weights klasoru ayri Drive linklerinden indirilir. Test gorselleri `/content/lejanter_doga_vlm_codex/data/raw/drive_test`, weight dosyalari `drive_weights/` altina kopyalanir.

In [ ]:
from pathlib import Path
import shutil

TEST_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1QJD49ylJs9PpidPDQEcCMhx0RltWUfTT?usp=sharing'
WEIGHTS_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1vmSSPsnu4fVBkM0yn1ScU7DFgDXyOksa?usp=sharing'
TEST_DOWNLOAD_DIR = Path('/content/shared_drive_test')
WEIGHTS_DOWNLOAD_DIR = Path('/content/shared_drive_weights')
PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')

for folder in [TEST_DOWNLOAD_DIR, WEIGHTS_DOWNLOAD_DIR]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

!pip install -q gdown
!gdown --folder '{TEST_DRIVE_FOLDER_URL}' -O /content/shared_drive_test --remaining-ok
!gdown --folder '{WEIGHTS_DRIVE_FOLDER_URL}' -O /content/shared_drive_weights --remaining-ok

print('Downloaded test files:')
for path in sorted(TEST_DOWNLOAD_DIR.rglob('*')):
    print(path)
print('Downloaded weight files:')
for path in sorted(WEIGHTS_DOWNLOAD_DIR.rglob('*')):
    print(path)

weights_dst = PROJECT_DIR / 'drive_weights'
if weights_dst.exists():
    shutil.rmtree(weights_dst)
weights_dst.mkdir(parents=True, exist_ok=True)
for pt in sorted(WEIGHTS_DOWNLOAD_DIR.rglob('*.pt')):
    dst = weights_dst / pt.name
    if dst.exists():
        dst = weights_dst / f'{pt.parent.name}-{pt.name}'
    shutil.copy2(pt, dst)
print('Weights copied to:', weights_dst)
for pt in sorted(weights_dst.rglob('*.pt')):
    print('-', pt)

test_src = TEST_DOWNLOAD_DIR / 'test'
if not test_src.exists():
    test_src = TEST_DOWNLOAD_DIR
test_dst = PROJECT_DIR / 'data/raw/drive_test'
if test_dst.exists():
    shutil.rmtree(test_dst)
test_dst.mkdir(parents=True, exist_ok=True)
for image in test_src.rglob('*'):
    if image.suffix.lower() in {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff','.avif'}:
        shutil.copy2(image, test_dst / image.name)
print('Test images copied to:', test_dst)
for image in sorted(test_dst.glob('*')):
    print('-', image)
assert list(weights_dst.rglob('*.pt')), 'Weight dosyasi bulunamadi. WEIGHTS_DRIVE_FOLDER_URL paylasim iznini kontrol et.'
assert list(test_dst.glob('*')), 'Test gorseli bulunamadi. TEST_DRIVE_FOLDER_URL icindeki test klasorunu kontrol et.'

## 3. Dataset Zip Varsa Ac ve Split Hazirla

Yeni egitim yapacaksan calistir. Sadece Drive weight ile test yapacaksan bu hucreyi atlayabilirsin.

In [ ]:
from pathlib import Path
import shutil, yaml

PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
zip_candidates = list(TEST_DOWNLOAD_DIR.rglob('*.zip')) + list(WEIGHTS_DOWNLOAD_DIR.rglob('*.zip'))
assert zip_candidates, 'Dataset zip bulunamadi. Egitim icin Drive dataset klasorune zip koy veya manuel yukle.'
zip_name = str(zip_candidates[0])
print('Dataset zip:', zip_name)

dataset_root = PROJECT_DIR / 'data/roboflow/gtu-mimari-lejant'
if dataset_root.exists():
    shutil.rmtree(dataset_root)
dataset_root.mkdir(parents=True, exist_ok=True)
!unzip -q "{zip_name}" -d /content/lejanter_doga_vlm_codex/data/roboflow/gtu-mimari-lejant

def image_count(split):
    p = dataset_root / split / 'images'
    return 0 if not p.exists() else sum(1 for x in p.iterdir() if x.suffix.lower() in {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff','.avif'})

if image_count('valid') == 0 or image_count('test') == 0:
    shutil.rmtree(dataset_root / 'valid', ignore_errors=True)
    shutil.rmtree(dataset_root / 'test', ignore_errors=True)
    !python scripts/split_yolo_dataset.py --root /content/lejanter_doga_vlm_codex/data/roboflow/gtu-mimari-lejant --valid 0.1 --test 0.1
print('Split counts:', {s: image_count(s) for s in ['train','valid','test']})

cfg = yaml.safe_load((PROJECT_DIR / 'configs/elements_dataset.yaml').read_text())
cfg['path'] = str(dataset_root)
colab_cfg = PROJECT_DIR / 'configs/elements_colab.yaml'
colab_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print(colab_cfg.read_text())

## 4. YOLO Segmentation Egitimi

In [ ]:
!cd /content/lejanter_doga_vlm_codex && yolo segment train \
  model=yolo11s-seg.pt \
  data=/content/lejanter_doga_vlm_codex/configs/elements_colab.yaml \
  project=/content/lejanter_doga_vlm_codex/outputs/runs \
  name=elements-seg-v3-no-erasing \
  epochs=300 \
  patience=150 \
  imgsz=1024 \
  batch=-1 \
  device=0 \
  erasing=0.0

## 5. Drive Weight ile Test Inference

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
weights_dir = PROJECT_DIR / 'drive_weights'
candidates = list(weights_dir.rglob('elements-seg-v3-no-erasing-best.pt')) or list(weights_dir.rglob('*v3*best*.pt')) or list(weights_dir.rglob('*.pt'))
assert candidates, 'Weight dosyasi bulunamadi. 2. hucreyi tekrar calistir.'
weight_path = candidates[0]
test_source = PROJECT_DIR / 'data/raw/drive_test'
print('Using weight:', weight_path)
print('Test source:', test_source)

!rm -rf /content/lejanter_doga_vlm_codex/outputs/runs/drive-test-v3-no-erasing
!cd /content/lejanter_doga_vlm_codex && python scripts/infer_yolo.py \
  --weights {str(weight_path)} \
  --source {str(test_source)} \
  --task segment \
  --out /content/lejanter_doga_vlm_codex/outputs/reports/drive_test_v3-no-erasing.json \
  --name drive-test-v3-no-erasing \
  --project /content/lejanter_doga_vlm_codex/outputs/runs \
  --save-visuals \
  --device 0

## 6. Sonuclari Goster

In [ ]:
from IPython.display import Image, display
from pathlib import Path
pred_dir = Path('/content/lejanter_doga_vlm_codex/outputs/runs/drive-test-v3-no-erasing')
for image_path in sorted(pred_dir.glob('*')):
    if image_path.suffix.lower() in {'.jpg','.jpeg','.png'}:
        display(Image(filename=str(image_path)))

## 7. Raporlari Drive'a Kopyala

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, datetime

drive.mount('/content/drive')
REPORTS_DIR = Path('/content/drive/MyDrive/_doganalci_onedrive/codes_doga_doktora_2025/lejant_vllm_sehemntaton_report_may_2026/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
export_dir = REPORTS_DIR / f'yolo_drive_test_{stamp}'
export_dir.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
for item in [PROJECT_DIR/'outputs/runs/drive-test-v3-no-erasing', PROJECT_DIR/'outputs/reports', PROJECT_DIR/'drive_weights']:
    if item.exists():
        dst = export_dir / item.name
        if item.is_dir():
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)
zip_path = shutil.make_archive(str(export_dir), 'zip', root_dir=REPORTS_DIR, base_dir=export_dir.name)
print('Reports copied to:', export_dir)
print('Zip:', zip_path)